# CNN Model v2

## Objective

This notebook develops and evaluates a revised CNN modelling pipeline using the deduplicated dataset produced in Step 3 v2.

The previous modelling workflow was based on the original 3,477 image records. Extended data-quality analysis subsequently identified substantial exact duplication and duplicate leakage across the original train, validation and test subsets.

The revised modelling stage therefore uses the leakage-controlled dataset of 2,407 unique images and the fixed stratified train/validation/test split. A baseline CNN is first established, followed by a series of controlled hyperparameter experiments investigating batch size, class weighting, learning rate and dropout.

Model selection is based on validation performance, while the test subset remains untouched for final evaluation.

## Reproducibility and configuration

A fixed random seed is used to make model initialisation, dataset shuffling and augmentation as reproducible as possible.

In [1]:
from pathlib import Path
import sys
import pickle

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

I0000 00:00:1789897776.722652    8062 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1789897776.801066    8062 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1789897778.717647    8062 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


In [2]:
def find_project_root(start: Path) -> Path:
    """Find the repository root from either the project or notebooks folder."""
    start = start.resolve()

    for candidate in (start, *start.parents):
        if (
            (candidate / "src" / "__init__.py").exists()
            and (candidate / "data").exists()
        ):
            return candidate

    raise FileNotFoundError(
        "Could not find the project root."
    )


PROJECT_ROOT = find_project_root(Path.cwd())

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from src import RANDOM_SEED

tf.keras.utils.set_random_seed(RANDOM_SEED)

print("TensorFlow version:", tf.__version__)
print("Project root:", PROJECT_ROOT)
print("Random seed:", RANDOM_SEED)

TensorFlow version: 2.21.0
Project root: /home/chetan/Desktop/DL_Project/grapevine-disease-classification
Random seed: 42


In [3]:
RAW_DATA_DIR = (
    PROJECT_ROOT
    / "data"
    / "raw"
    / "GVLiD"
)

SPLIT_PATH = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "dataset_split.csv"
)

RESULTS_DIR = PROJECT_ROOT / "results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

BEST_MODEL_PATH = (
    RESULTS_DIR
    / "cnn_v2_baseline_best.keras"
)

FINAL_MODEL_PATH = (
    RESULTS_DIR
    / "cnn_v2_baseline_final.keras"
)

HISTORY_PATH = (
    RESULTS_DIR
    / "cnn_v2_baseline_training_history.pkl"
)

TRAINING_LOG_PATH = (
    RESULTS_DIR
    / "cnn_v2_baseline_training_log.csv"
)

In [4]:
print("Raw data exists:", RAW_DATA_DIR.exists())
print("Split file exists:", SPLIT_PATH.exists())
print("Results directory:", RESULTS_DIR)

Raw data exists: True
Split file exists: True
Results directory: /home/chetan/Desktop/DL_Project/grapevine-disease-classification/results


## Model configuration

The baseline CNN uses 128 × 128 RGB images with a batch size of 32. The four target classes are encoded consistently throughout training and evaluation.

These baseline settings are preserved unless a specific hyperparameter experiment explicitly changes one of them.

In [5]:
IMG_SIZE = (128, 128)
BATCH_SIZE = 32

CLASS_NAMES = [
    "Black Rot",
    "Esca",
    "Healthy",
    "Leaf Blight",
]

NUM_CLASSES = len(CLASS_NAMES)

CLASS_TO_INDEX = {
    class_name: index
    for index, class_name in enumerate(CLASS_NAMES)
}

INDEX_TO_CLASS = {
    index: class_name
    for class_name, index in CLASS_TO_INDEX.items()
}

## Dataset loading

The fixed train, validation and test allocation created during the revised EDA is loaded from `dataset_split.csv`. The data is not split again during modelling.

In [6]:
split_df = pd.read_csv(SPLIT_PATH)

print("Dataset shape:", split_df.shape)

print("\nColumns:")
print(split_df.columns.tolist())

display(split_df.head())

Dataset shape: (2407, 5)

Columns:
['image_filename', 'sha256', 'Target_Class', 'Vineyard', 'Split']


,image_filename,sha256,Target_Class,Vineyard,Split
0,healthy267.jpg,7a89f052a9b5c4a77ef9b6130b5f84d65a7da8218d738b...,Healthy,Vineyard_3,Train
1,black rot677.jpg,74f16e0042eecbc2b3de8418089eb4a1fc7266ef9a85d9...,Black Rot,Vineyard_25,Train
2,leaf blight431.jpg,856203b686272c318b09533e5ed7c6535ef8a37688b2ca...,Leaf Blight,Vineyard_8,Train
3,healthy413.jpg,d1d02735cc5304ee6569bb7e8b1eff34dcc9c6fd97574c...,Healthy,Vineyard_2,Train
4,esca675.jpg,f2486b88ab87fbceaf327f7ae0ff24857e1a4d3d012004...,Esca,Vineyard_24,Train


In [7]:
print("Total records:", len(split_df))
print("Unique SHA-256 hashes:", split_df["sha256"].nunique())

print("\nImages per split:")
print(split_df["Split"].value_counts())

print("\nImages per class:")
print(split_df["Target_Class"].value_counts())

Total records: 2407
Unique SHA-256 hashes: 2407

Images per split:
Split
Train         1684
Test           362
Validation     361
Name: count, dtype: int64

Images per class:
Target_Class
Healthy        1093
Esca            886
Leaf Blight     331
Black Rot        97
Name: count, dtype: int64


In [8]:
class_split_table = pd.crosstab(
    split_df["Split"],
    split_df["Target_Class"]
)

class_split_table = class_split_table.reindex(
    columns=CLASS_NAMES
)

display(class_split_table)

Target_Class,Black Rot,Esca,Healthy,Leaf Blight
Split,,,,
Test,15,133,164,50
Train,68,620,765,231
Validation,14,133,164,50


In [9]:
CLASS_TO_FOLDER = {
    "Black Rot": "Black rot",
    "Esca": "esca",
    "Healthy": "healthy",
    "Leaf Blight": "leaf blight",
}

split_df["image_path"] = split_df.apply(
    lambda row: str(
        RAW_DATA_DIR
        / CLASS_TO_FOLDER[row["Target_Class"]]
        / row["image_filename"]
    ),
    axis=1
)

display(
    split_df[
        [
            "image_filename",
            "Target_Class",
            "Split",
            "image_path",
        ]
    ].head()
)

,image_filename,Target_Class,Split,image_path
0,healthy267.jpg,Healthy,Train,/home/chetan/Desktop/DL_Project/grapevine-dise...
1,black rot677.jpg,Black Rot,Train,/home/chetan/Desktop/DL_Project/grapevine-dise...
2,leaf blight431.jpg,Leaf Blight,Train,/home/chetan/Desktop/DL_Project/grapevine-dise...
3,healthy413.jpg,Healthy,Train,/home/chetan/Desktop/DL_Project/grapevine-dise...
4,esca675.jpg,Esca,Train,/home/chetan/Desktop/DL_Project/grapevine-dise...


In [10]:
split_df["file_exists"] = (
    split_df["image_path"]
    .apply(lambda path: Path(path).exists())
)

missing_images = split_df[
    ~split_df["file_exists"]
]

print("Missing modelling images:", len(missing_images))

if len(missing_images) > 0:
    display(
        missing_images[
            [
                "image_filename",
                "Target_Class",
                "image_path",
            ]
        ].head(20)
    )

Missing modelling images: 0


In [11]:
split_df["label"] = (
    split_df["Target_Class"]
    .map(CLASS_TO_INDEX)
)

assert split_df["label"].notna().all()

split_df["label"] = split_df["label"].astype(int)

display(
    split_df[
        ["Target_Class", "label"]
    ]
    .drop_duplicates()
    .sort_values("label")
)

,Target_Class,label
1,Black Rot,0
4,Esca,1
0,Healthy,2
2,Leaf Blight,3


In [12]:
train_df = (
    split_df[split_df["Split"] == "Train"]
    .copy()
    .reset_index(drop=True)
)

validation_df = (
    split_df[split_df["Split"] == "Validation"]
    .copy()
    .reset_index(drop=True)
)

test_df = (
    split_df[split_df["Split"] == "Test"]
    .copy()
    .reset_index(drop=True)
)

print("Train:", len(train_df))
print("Validation:", len(validation_df))
print("Test:", len(test_df))

Train: 1684
Validation: 361
Test: 362


## Class imbalance

The deduplicated training set remains imbalanced, particularly for Black Rot. Class weights are therefore calculated from the training subset so that errors on minority classes contribute more strongly to the baseline training loss.

Class weighting is retained for the baseline and most experiments. A separate controlled experiment removes the class weights to assess their effect on validation performance.

In [13]:
class_counts = (
    train_df["label"]
    .value_counts()
    .sort_index()
)

total_samples = len(train_df)
num_classes = len(class_counts)

CLASS_WEIGHTS = {
    int(class_index): total_samples / (num_classes * count)
    for class_index, count in class_counts.items()
}

print("Training class counts:")
print(class_counts)

print("\nClass weights:")
for class_index, weight in CLASS_WEIGHTS.items():
    print(
        f"{class_index} - {INDEX_TO_CLASS[class_index]}: "
        f"{weight:.4f}"
    )

Training class counts:
label
0     68
1    620
2    765
3    231
Name: count, dtype: int64

Class weights:
0 - Black Rot: 6.1912
1 - Esca: 0.6790
2 - Healthy: 0.5503
3 - Leaf Blight: 1.8225


## Image preprocessing

Images are resized to 128 × 128 pixels, converted to RGB tensors and normalised to the [0, 1] range before being provided to the CNN.

In [15]:
AUTOTUNE = tf.data.AUTOTUNE


def load_and_preprocess_image(image_path, label):
    image_bytes = tf.io.read_file(image_path)

    image = tf.io.decode_image(
        image_bytes,
        channels=3,
        expand_animations=False
    )

    image.set_shape([None, None, 3])

    image = tf.image.resize(
        image,
        IMG_SIZE
    )

    image = tf.cast(
        image,
        tf.float32
    ) / 255.0

    return image, label

In [16]:
def create_dataset(df, training=False):
    image_paths = df["image_path"].to_numpy()
    labels = df["label"].to_numpy()

    dataset = tf.data.Dataset.from_tensor_slices(
        (image_paths, labels)
    )

    if training:
        dataset = dataset.shuffle(
            buffer_size=len(df),
            seed=RANDOM_SEED,
            reshuffle_each_iteration=True
        )

    dataset = dataset.map(
        load_and_preprocess_image,
        num_parallel_calls=AUTOTUNE
    )

    dataset = dataset.batch(
        BATCH_SIZE
    )

    dataset = dataset.prefetch(
        AUTOTUNE
    )

    return dataset

In [17]:
train_ds = create_dataset(
    train_df,
    training=True
)

validation_ds = create_dataset(
    validation_df,
    training=False
)

E0000 00:00:1789897837.742302    8062 cuda_platform.cc:52] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


In [18]:
for images, labels in train_ds.take(1):
    print("Image batch shape:", images.shape)
    print("Label batch shape:", labels.shape)
    print("Image dtype:", images.dtype)
    print(
        "Pixel range:",
        float(tf.reduce_min(images)),
        "to",
        float(tf.reduce_max(images))
    )

Image batch shape: (32, 128, 128, 3)
Label batch shape: (32,)
Image dtype: <dtype: 'float32'>
Pixel range: 0.0 to 1.0


## Data augmentation

Moderate geometric augmentation is applied only during training. Horizontal flips, small rotations and small zoom transformations increase the visual diversity of the training data while preserving the underlying disease class.

Validation images are not randomly augmented so that model selection remains deterministic. The test subset is reserved for the separate final evaluation stage.

In [19]:
data_augmentation = keras.Sequential(
    [
        layers.RandomFlip(
            "horizontal",
            seed=RANDOM_SEED
        ),
        layers.RandomRotation(
            0.10,
            seed=RANDOM_SEED
        ),
        layers.RandomZoom(
            0.10,
            seed=RANDOM_SEED
        ),
    ],
    name="data_augmentation"
)

## Baseline CNN architecture

A conventional convolutional neural network is trained from scratch as the leakage-controlled baseline.

Three convolutional blocks progressively learn spatial image features, while MaxPooling reduces the spatial dimensions of the feature maps. Global average pooling and dropout are used before the final dense classification layers to reduce model complexity and help limit overfitting.

In [20]:
model = keras.Sequential(
    [
        layers.Input(
            shape=(
                IMG_SIZE[0],
                IMG_SIZE[1],
                3
            )
        ),

        data_augmentation,

        layers.Conv2D(
            32,
            kernel_size=3,
            activation="relu",
            padding="same"
        ),
        layers.MaxPooling2D(),

        layers.Conv2D(
            64,
            kernel_size=3,
            activation="relu",
            padding="same"
        ),
        layers.MaxPooling2D(),

        layers.Conv2D(
            128,
            kernel_size=3,
            activation="relu",
            padding="same"
        ),
        layers.MaxPooling2D(),

        layers.GlobalAveragePooling2D(),

        layers.Dropout(0.30),

        layers.Dense(
            64,
            activation="relu"
        ),

        layers.Dropout(0.30),

        layers.Dense(
            NUM_CLASSES,
            activation="softmax"
        ),
    ],
    name="cnn_v2_baseline"
)

In [21]:
model.summary()

Model: "cnn_v2_baseline"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ data_augmentation (Sequential)  │ (None, 128, 128, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d (Conv2D)                 │ (None, 128, 128, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 64, 64, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 64, 64, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 32, 32, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 32, 32, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 16, 16, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 128)            │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 4)              │           260 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 101,764 (397.52 KB)

 Trainable params: 101,764 (397.52 KB)

 Non-trainable params: 0 (0.00 B)

## Model compilation

The CNN uses the Adam optimiser with sparse categorical cross-entropy loss. Sparse categorical cross-entropy is appropriate because the four target classes are represented as integer labels.

In [22]:
model.compile(
    optimizer=keras.optimizers.Adam(
        learning_rate=0.001
    ),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

## Training strategy

The baseline CNN and all controlled experiments are trained using the fixed deduplicated training subset and monitored against the same validation subset after each epoch.

Early stopping limits unnecessary training once validation loss stops improving. Learning-rate reduction lowers the learning rate when validation performance plateaus, while model checkpointing preserves the model with the lowest validation loss.

Class weights are applied to the baseline and all experiments except the dedicated no-class-weights experiment.

Each hyperparameter experiment changes one modelling choice at a time while preserving the remaining training configuration. Validation loss is used consistently as the primary model-selection criterion.

The test subset is not used during training or hyperparameter selection and remains reserved for final evaluation.

In [23]:
callbacks = [
    keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=5,
        restore_best_weights=True,
        verbose=1
    ),

    keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=2,
        min_lr=1e-6,
        verbose=1
    ),

    keras.callbacks.ModelCheckpoint(
        filepath=BEST_MODEL_PATH,
        monitor="val_loss",
        save_best_only=True,
        verbose=1
    ),

    keras.callbacks.CSVLogger(
        TRAINING_LOG_PATH
    )
]

In [24]:
MAX_EPOCHS = 50

history = model.fit(
    train_ds,
    validation_data=validation_ds,
    epochs=MAX_EPOCHS,
    class_weight=CLASS_WEIGHTS,
    callbacks=callbacks,
    shuffle=False
)

Epoch 1/50
53/53 ━━━━━━━━━━━━━━━━━━━━ 0s 798ms/step - accuracy: 0.4400 - loss: 1.3333
Epoch 1: val_loss improved from None to 0.97825, saving model to /home/chetan/Desktop/DL_Project/grapevine-disease-classification/results/cnn_v2_baseline_best.keras

Epoch 1: finished saving model to /home/chetan/Desktop/DL_Project/grapevine-disease-classification/results/cnn_v2_baseline_best.keras
53/53 ━━━━━━━━━━━━━━━━━━━━ 50s 873ms/step - accuracy: 0.4400 - loss: 1.3333 - val_accuracy: 0.5263 - val_loss: 0.9783 - learning_rate: 0.0010
Epoch 2/50
53/53 ━━━━━━━━━━━━━━━━━━━━ 0s 889ms/step - accuracy: 0.4941 - loss: 1.1721
Epoch 2: val_loss improved from 0.97825 to 0.90404, saving model to /home/chetan/Desktop/DL_Project/grapevine-disease-classification/results/cnn_v2_baseline_best.keras

Epoch 2: finished saving model to /home/chetan/Desktop/DL_Project/grapevine-disease-classification/results/cnn_v2_baseline_best.keras
53/53 ━━━━━━━━━━━━━━━━━━━━ 52s 959ms/step - accuracy: 0.4941 - loss: 1.1721 - val_a

KeyboardInterrupt: 

In [ ]:
model.save(FINAL_MODEL_PATH)

with open(HISTORY_PATH, "wb") as f:
    pickle.dump(history.history, f)

print("Best checkpoint saved to:", BEST_MODEL_PATH)
print("Final restored model saved to:", FINAL_MODEL_PATH)
print("Training history saved to:", HISTORY_PATH)
print("Training log saved to:", TRAINING_LOG_PATH)

In [ ]:
plt.figure(figsize=(8, 5))

plt.plot(
    history.history["accuracy"],
    label="Training accuracy"
)

plt.plot(
    history.history["val_accuracy"],
    label="Validation accuracy"
)

plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("CNN v2 Training and Validation Accuracy")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

In [ ]:
plt.figure(figsize=(8, 5))

plt.plot(
    history.history["loss"],
    label="Training loss"
)

plt.plot(
    history.history["val_loss"],
    label="Validation loss"
)

plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("CNN v2 Training and Validation Loss")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

In [ ]:
best_epoch = (
    np.argmin(history.history["val_loss"]) + 1
)

best_val_loss = min(
    history.history["val_loss"]
)

best_val_accuracy = history.history[
    "val_accuracy"
][best_epoch - 1]

print("Best epoch:", best_epoch)
print(f"Best validation loss: {best_val_loss:.4f}")
print(
    f"Validation accuracy at best epoch: "
    f"{best_val_accuracy:.4f}"
)

## Hyperparameter experiments

A set of controlled experiments is conducted to investigate the effect of selected training hyperparameters on validation performance.

Only one modelling choice is changed at a time while the fixed training and validation subsets, preprocessing pipeline, augmentation strategy and random seed are preserved.

The test subset remains untouched and is reserved for the final evaluation stage.

In [ ]:
def create_experiment_dataset(
    df,
    training=False,
    batch_size=32
):
    image_paths = df["image_path"].to_numpy()
    labels = df["label"].to_numpy()

    dataset = tf.data.Dataset.from_tensor_slices(
        (image_paths, labels)
    )

    if training:
        dataset = dataset.shuffle(
            buffer_size=len(df),
            seed=RANDOM_SEED,
            reshuffle_each_iteration=True
        )

    dataset = dataset.map(
        load_and_preprocess_image,
        num_parallel_calls=AUTOTUNE
    )

    dataset = dataset.batch(batch_size)

    dataset = dataset.prefetch(AUTOTUNE)

    return dataset

In [ ]:
train_ds_bs1 = create_experiment_dataset(
    train_df,
    training=True,
    batch_size=1
)

# Keep validation batching identical to the baseline
validation_ds_exp = validation_ds

### Experiment 1 — Batch size 1

The training batch size is reduced from 32 to 1 while all other modelling choices are kept unchanged. Class weights remain enabled.

This experiment investigates the effect of highly stochastic gradient updates on convergence and validation performance.

In [ ]:
BS1_BEST_MODEL_PATH = (
    RESULTS_DIR / "cnn_v2_batch1_best.keras"
)

BS1_FINAL_MODEL_PATH = (
    RESULTS_DIR / "cnn_v2_batch1_final.keras"
)

BS1_HISTORY_PATH = (
    RESULTS_DIR / "cnn_v2_batch1_training_history.pkl"
)

BS1_LOG_PATH = (
    RESULTS_DIR / "cnn_v2_batch1_training_log.csv"
)

print("Batch size 1 experiment paths configured.")

In [ ]:
tf.keras.utils.set_random_seed(RANDOM_SEED)

bs1_augmentation = keras.Sequential(
    [
        layers.RandomFlip(
            "horizontal",
            seed=RANDOM_SEED
        ),
        layers.RandomRotation(
            0.10,
            seed=RANDOM_SEED
        ),
        layers.RandomZoom(
            0.10,
            seed=RANDOM_SEED
        ),
    ],
    name="bs1_data_augmentation"
)

bs1_model = keras.Sequential(
    [
        layers.Input(
            shape=(*IMG_SIZE, 3)
        ),

        bs1_augmentation,

        layers.Conv2D(
            32,
            kernel_size=3,
            activation="relu",
            padding="same"
        ),
        layers.MaxPooling2D(),

        layers.Conv2D(
            64,
            kernel_size=3,
            activation="relu",
            padding="same"
        ),
        layers.MaxPooling2D(),

        layers.Conv2D(
            128,
            kernel_size=3,
            activation="relu",
            padding="same"
        ),
        layers.MaxPooling2D(),

        layers.GlobalAveragePooling2D(),

        layers.Dropout(0.30),

        layers.Dense(
            64,
            activation="relu"
        ),

        layers.Dropout(0.30),

        layers.Dense(
            NUM_CLASSES,
            activation="softmax"
        ),
    ],
    name="cnn_v2_batch1"
)

In [ ]:
bs1_model.compile(
    optimizer=keras.optimizers.Adam(
        learning_rate=0.001
    ),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

In [ ]:
bs1_model.summary()

In [ ]:
bs1_callbacks = [
    keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=5,
        restore_best_weights=True,
        verbose=1
    ),

    keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=2,
        min_lr=1e-6,
        verbose=1
    ),

    keras.callbacks.ModelCheckpoint(
        filepath=BS1_BEST_MODEL_PATH,
        monitor="val_loss",
        save_best_only=True,
        verbose=1
    ),

    keras.callbacks.CSVLogger(
        BS1_LOG_PATH
    )
]

In [ ]:
bs1_history = bs1_model.fit(
    train_ds_bs1,
    validation_data=validation_ds_exp,
    epochs=MAX_EPOCHS,
    class_weight=CLASS_WEIGHTS,
    callbacks=bs1_callbacks,
    shuffle=False
)

In [ ]:
bs1_model.save(BS1_FINAL_MODEL_PATH)

with open(BS1_HISTORY_PATH, "wb") as f:
    pickle.dump(bs1_history.history, f)

print("Best checkpoint saved to:", BS1_BEST_MODEL_PATH)
print("Final restored model saved to:", BS1_FINAL_MODEL_PATH)
print("Training history saved to:", BS1_HISTORY_PATH)
print("Training log saved to:", BS1_LOG_PATH)

In [ ]:
bs1_best_epoch = (
    np.argmin(bs1_history.history["val_loss"]) + 1
)

bs1_best_val_loss = min(
    bs1_history.history["val_loss"]
)

bs1_best_val_accuracy = bs1_history.history[
    "val_accuracy"
][bs1_best_epoch - 1]

print("Batch size 1 experiment")
print("Best epoch:", bs1_best_epoch)
print(f"Best validation loss: {bs1_best_val_loss:.4f}")
print(
    f"Validation accuracy at best epoch: "
    f"{bs1_best_val_accuracy:.4f}"
)

In [ ]:
comparison_exp1_df = pd.DataFrame(
    [
        {
            "Model": "Baseline",
            "Batch size": 32,
            "Class weights": "Yes",
            "Best epoch": best_epoch,
            "Val loss": best_val_loss,
            "Val accuracy": best_val_accuracy,
        },
        {
            "Model": "Batch size 1",
            "Batch size": 1,
            "Class weights": "Yes",
            "Best epoch": bs1_best_epoch,
            "Val loss": bs1_best_val_loss,
            "Val accuracy": bs1_best_val_accuracy,
        },
    ]
)

comparison_exp1_df["Val loss"] = (
    comparison_exp1_df["Val loss"].round(4)
)

comparison_exp1_df["Val accuracy"] = (
    comparison_exp1_df["Val accuracy"].round(4)
)

comparison_exp1_df

#### Experiment 1 result

Reducing the training batch size from 32 to 1 did not improve validation performance. The batch-size-1 model reached its best validation loss of 0.4472 at epoch 16, with a validation accuracy of 80.61%.

This was slightly worse than the baseline model, which achieved a validation loss of 0.4333 and validation accuracy of 83.66%. The smaller batch size also substantially increased the number of gradient updates per epoch, making training less computationally efficient.

For this dataset and CNN architecture, a batch size of 32 therefore provided a better validation result than a batch size of 1.

### Experiment 2 — No class weights

The baseline CNN is retrained without class weights while the batch size, architecture, preprocessing, augmentation, learning rate and validation procedure remain unchanged.

This experiment investigates whether class weighting improves validation performance when training on the imbalanced dataset.

In [ ]:
NW_BEST_MODEL_PATH = (
    RESULTS_DIR / "cnn_v2_no_weights_best.keras"
)

NW_FINAL_MODEL_PATH = (
    RESULTS_DIR / "cnn_v2_no_weights_final.keras"
)

NW_HISTORY_PATH = (
    RESULTS_DIR / "cnn_v2_no_weights_training_history.pkl"
)

NW_LOG_PATH = (
    RESULTS_DIR / "cnn_v2_no_weights_training_log.csv"
)

print("No-class-weights experiment paths configured.")

In [40]:
tf.keras.utils.set_random_seed(RANDOM_SEED)

nw_augmentation = keras.Sequential(
    [
        layers.RandomFlip(
            "horizontal",
            seed=RANDOM_SEED
        ),
        layers.RandomRotation(
            0.10,
            seed=RANDOM_SEED
        ),
        layers.RandomZoom(
            0.10,
            seed=RANDOM_SEED
        ),
    ],
    name="nw_data_augmentation"
)

nw_model = keras.Sequential(
    [
        layers.Input(
            shape=(*IMG_SIZE, 3)
        ),

        nw_augmentation,

        layers.Conv2D(
            32,
            kernel_size=3,
            activation="relu",
            padding="same"
        ),
        layers.MaxPooling2D(),

        layers.Conv2D(
            64,
            kernel_size=3,
            activation="relu",
            padding="same"
        ),
        layers.MaxPooling2D(),

        layers.Conv2D(
            128,
            kernel_size=3,
            activation="relu",
            padding="same"
        ),
        layers.MaxPooling2D(),

        layers.GlobalAveragePooling2D(),

        layers.Dropout(0.30),

        layers.Dense(
            64,
            activation="relu"
        ),

        layers.Dropout(0.30),

        layers.Dense(
            NUM_CLASSES,
            activation="softmax"
        ),
    ],
    name="cnn_v2_no_weights"
)

In [ ]:
nw_model.compile(
    optimizer=keras.optimizers.Adam(
        learning_rate=0.001
    ),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

In [ ]:
nw_callbacks = [
    keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=5,
        restore_best_weights=True,
        verbose=1
    ),

    keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=2,
        min_lr=1e-6,
        verbose=1
    ),

    keras.callbacks.ModelCheckpoint(
        filepath=NW_BEST_MODEL_PATH,
        monitor="val_loss",
        save_best_only=True,
        verbose=1
    ),

    keras.callbacks.CSVLogger(
        NW_LOG_PATH
    )
]

In [ ]:
nw_history = nw_model.fit(
    train_ds,
    validation_data=validation_ds,
    epochs=MAX_EPOCHS,
    callbacks=nw_callbacks,
    shuffle=False
)

In [ ]:
nw_model.save(NW_FINAL_MODEL_PATH)

with open(NW_HISTORY_PATH, "wb") as f:
    pickle.dump(nw_history.history, f)

print("Best checkpoint saved to:", NW_BEST_MODEL_PATH)
print("Final restored model saved to:", NW_FINAL_MODEL_PATH)
print("Training history saved to:", NW_HISTORY_PATH)
print("Training log saved to:", NW_LOG_PATH)

In [ ]:
nw_best_epoch = (
    np.argmin(nw_history.history["val_loss"]) + 1
)

nw_best_val_loss = min(
    nw_history.history["val_loss"]
)

nw_best_val_accuracy = nw_history.history[
    "val_accuracy"
][nw_best_epoch - 1]

print("No class weights experiment")
print("Best epoch:", nw_best_epoch)
print(f"Best validation loss: {nw_best_val_loss:.4f}")
print(
    f"Validation accuracy at best epoch: "
    f"{nw_best_val_accuracy:.4f}"
)

In [ ]:
comparison_df = pd.DataFrame(
    [
        {
            "Model": "Baseline",
            "Batch size": 32,
            "Class weights": "Yes",
            "Best epoch": best_epoch,
            "Val loss": best_val_loss,
            "Val accuracy": best_val_accuracy,
        },
        {
            "Model": "Batch size 1",
            "Batch size": 1,
            "Class weights": "Yes",
            "Best epoch": bs1_best_epoch,
            "Val loss": bs1_best_val_loss,
            "Val accuracy": bs1_best_val_accuracy,
        },
        {
            "Model": "No class weights",
            "Batch size": 32,
            "Class weights": "No",
            "Best epoch": nw_best_epoch,
            "Val loss": nw_best_val_loss,
            "Val accuracy": nw_best_val_accuracy,
        },
    ]
)

comparison_df["Val loss"] = (
    comparison_df["Val loss"].round(4)
)

comparison_df["Val accuracy"] = (
    comparison_df["Val accuracy"].round(4)
)

comparison_df

### Experiment 2 result

Removing class weights produced the same validation accuracy as the baseline model (83.66%), but resulted in a slightly higher validation loss of 0.4474 compared with 0.4333 for the weighted baseline.

This suggests that class weighting did not change overall validation accuracy in this experiment, but the weighted model achieved a slightly better validation loss. Because overall accuracy can hide differences in performance between majority and minority classes, the effect of class weighting should also be examined using class-level precision, recall, F1-score and the confusion matrix during final evaluation.

### Experiment 3 — Lower learning rate

The initial learning rate is reduced from 0.001 to 0.0005 while all other modelling choices remain unchanged.

This experiment investigates whether a smaller initial learning rate produces more stable optimisation and improved validation performance.

In [ ]:
LR_BEST_MODEL_PATH = (
    RESULTS_DIR / "cnn_v2_lr0005_best.keras"
)

LR_FINAL_MODEL_PATH = (
    RESULTS_DIR / "cnn_v2_lr0005_final.keras"
)

LR_HISTORY_PATH = (
    RESULTS_DIR / "cnn_v2_lr0005_training_history.pkl"
)

LR_LOG_PATH = (
    RESULTS_DIR / "cnn_v2_lr0005_training_log.csv"
)

print("Learning-rate experiment paths configured.")

In [ ]:
tf.keras.utils.set_random_seed(RANDOM_SEED)

lr_augmentation = keras.Sequential(
    [
        layers.RandomFlip(
            "horizontal",
            seed=RANDOM_SEED
        ),
        layers.RandomRotation(
            0.10,
            seed=RANDOM_SEED
        ),
        layers.RandomZoom(
            0.10,
            seed=RANDOM_SEED
        ),
    ],
    name="lr_data_augmentation"
)

lr_model = keras.Sequential(
    [
        layers.Input(
            shape=(*IMG_SIZE, 3)
        ),

        lr_augmentation,

        layers.Conv2D(
            32,
            kernel_size=3,
            activation="relu",
            padding="same"
        ),
        layers.MaxPooling2D(),

        layers.Conv2D(
            64,
            kernel_size=3,
            activation="relu",
            padding="same"
        ),
        layers.MaxPooling2D(),

        layers.Conv2D(
            128,
            kernel_size=3,
            activation="relu",
            padding="same"
        ),
        layers.MaxPooling2D(),

        layers.GlobalAveragePooling2D(),

        layers.Dropout(0.30),

        layers.Dense(
            64,
            activation="relu"
        ),

        layers.Dropout(0.30),

        layers.Dense(
            NUM_CLASSES,
            activation="softmax"
        ),
    ],
    name="cnn_v2_lr0005"
)

In [ ]:
lr_model.compile(
    optimizer=keras.optimizers.Adam(
        learning_rate=0.0005
    ),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

In [ ]:
lr_callbacks = [
    keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=5,
        restore_best_weights=True,
        verbose=1
    ),

    keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=2,
        min_lr=1e-6,
        verbose=1
    ),

    keras.callbacks.ModelCheckpoint(
        filepath=LR_BEST_MODEL_PATH,
        monitor="val_loss",
        save_best_only=True,
        verbose=1
    ),

    keras.callbacks.CSVLogger(
        LR_LOG_PATH
    )
]

In [ ]:
lr_history = lr_model.fit(
    train_ds,
    validation_data=validation_ds,
    epochs=MAX_EPOCHS,
    class_weight=CLASS_WEIGHTS,
    callbacks=lr_callbacks,
    shuffle=False
)

In [ ]:
lr_model.save(LR_FINAL_MODEL_PATH)

with open(LR_HISTORY_PATH, "wb") as f:
    pickle.dump(lr_history.history, f)

print("Best checkpoint saved to:", LR_BEST_MODEL_PATH)
print("Final restored model saved to:", LR_FINAL_MODEL_PATH)
print("Training history saved to:", LR_HISTORY_PATH)
print("Training log saved to:", LR_LOG_PATH)

In [ ]:
lr_best_epoch = (
    np.argmin(lr_history.history["val_loss"]) + 1
)

lr_best_val_loss = min(
    lr_history.history["val_loss"]
)

lr_best_val_accuracy = lr_history.history[
    "val_accuracy"
][lr_best_epoch - 1]

print("Lower learning rate experiment")
print("Best epoch:", lr_best_epoch)
print(f"Best validation loss: {lr_best_val_loss:.4f}")
print(
    f"Validation accuracy at best epoch: "
    f"{lr_best_val_accuracy:.4f}"
)

In [ ]:
comparison_exp3_df = pd.DataFrame(
    [
        {
            "Model": "Baseline",
            "Batch size": 32,
            "Class weights": "Yes",
            "Learning rate": 0.001,
            "Best epoch": best_epoch,
            "Val loss": best_val_loss,
            "Val accuracy": best_val_accuracy,
        },
        {
            "Model": "Batch size 1",
            "Batch size": 1,
            "Class weights": "Yes",
            "Learning rate": 0.001,
            "Best epoch": bs1_best_epoch,
            "Val loss": bs1_best_val_loss,
            "Val accuracy": bs1_best_val_accuracy,
        },
        {
            "Model": "No class weights",
            "Batch size": 32,
            "Class weights": "No",
            "Learning rate": 0.001,
            "Best epoch": nw_best_epoch,
            "Val loss": nw_best_val_loss,
            "Val accuracy": nw_best_val_accuracy,
        },
        {
            "Model": "Lower learning rate",
            "Batch size": 32,
            "Class weights": "Yes",
            "Learning rate": 0.0005,
            "Best epoch": lr_best_epoch,
            "Val loss": lr_best_val_loss,
            "Val accuracy": lr_best_val_accuracy,
        },
    ]
)

comparison_exp3_df["Val loss"] = (
    comparison_exp3_df["Val loss"].round(4)
)

comparison_exp3_df["Val accuracy"] = (
    comparison_exp3_df["Val accuracy"].round(4)
)

comparison_exp3_df

#### Experiment 3 result

Reducing the initial learning rate from 0.001 to 0.0005 produced a slightly higher validation accuracy of 84.49%, compared with 83.66% for the baseline.

However, the best validation loss increased slightly from 0.4333 to 0.4431. Therefore, the lower learning rate improved overall validation accuracy but did not improve the validation-loss criterion used for model checkpointing and selection.

The result suggests that a smaller learning rate can produce a competitive model, although the baseline remains stronger according to validation loss.

### Experiment 4 — Increased dropout

The dropout rate is increased from 0.30 to 0.40 while all other modelling choices remain unchanged.

This experiment investigates whether stronger regularisation improves generalisation by reducing overfitting.

In [ ]:
DO_BEST_MODEL_PATH = (
    RESULTS_DIR / "cnn_v2_dropout040_best.keras"
)

DO_FINAL_MODEL_PATH = (
    RESULTS_DIR / "cnn_v2_dropout040_final.keras"
)

DO_HISTORY_PATH = (
    RESULTS_DIR / "cnn_v2_dropout040_training_history.pkl"
)

DO_LOG_PATH = (
    RESULTS_DIR / "cnn_v2_dropout040_training_log.csv"
)

print("Dropout experiment paths configured.")

In [ ]:
tf.keras.utils.set_random_seed(RANDOM_SEED)

do_augmentation = keras.Sequential(
    [
        layers.RandomFlip(
            "horizontal",
            seed=RANDOM_SEED
        ),
        layers.RandomRotation(
            0.10,
            seed=RANDOM_SEED
        ),
        layers.RandomZoom(
            0.10,
            seed=RANDOM_SEED
        ),
    ],
    name="dropout_data_augmentation"
)

do_model = keras.Sequential(
    [
        layers.Input(
            shape=(*IMG_SIZE, 3)
        ),

        do_augmentation,

        layers.Conv2D(
            32,
            kernel_size=3,
            activation="relu",
            padding="same"
        ),
        layers.MaxPooling2D(),

        layers.Conv2D(
            64,
            kernel_size=3,
            activation="relu",
            padding="same"
        ),
        layers.MaxPooling2D(),

        layers.Conv2D(
            128,
            kernel_size=3,
            activation="relu",
            padding="same"
        ),
        layers.MaxPooling2D(),

        layers.GlobalAveragePooling2D(),

        layers.Dropout(0.40),

        layers.Dense(
            64,
            activation="relu"
        ),

        layers.Dropout(0.40),

        layers.Dense(
            NUM_CLASSES,
            activation="softmax"
        ),
    ],
    name="cnn_v2_dropout040"
)

In [ ]:
do_model.compile(
    optimizer=keras.optimizers.Adam(
        learning_rate=0.001
    ),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

In [ ]:
do_callbacks = [
    keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=5,
        restore_best_weights=True,
        verbose=1
    ),

    keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=2,
        min_lr=1e-6,
        verbose=1
    ),

    keras.callbacks.ModelCheckpoint(
        filepath=DO_BEST_MODEL_PATH,
        monitor="val_loss",
        save_best_only=True,
        verbose=1
    ),

    keras.callbacks.CSVLogger(
        DO_LOG_PATH
    )
]

In [ ]:
do_history = do_model.fit(
    train_ds,
    validation_data=validation_ds,
    epochs=MAX_EPOCHS,
    class_weight=CLASS_WEIGHTS,
    callbacks=do_callbacks,
    shuffle=False
)

In [ ]:
do_model.save(DO_FINAL_MODEL_PATH)

with open(DO_HISTORY_PATH, "wb") as f:
    pickle.dump(do_history.history, f)

print("Best checkpoint saved to:", DO_BEST_MODEL_PATH)
print("Final restored model saved to:", DO_FINAL_MODEL_PATH)
print("Training history saved to:", DO_HISTORY_PATH)
print("Training log saved to:", DO_LOG_PATH)

In [ ]:
do_best_epoch = (
    np.argmin(do_history.history["val_loss"]) + 1
)

do_best_val_loss = min(
    do_history.history["val_loss"]
)

do_best_val_accuracy = do_history.history[
    "val_accuracy"
][do_best_epoch - 1]

print("Increased dropout experiment")
print("Best epoch:", do_best_epoch)
print(f"Best validation loss: {do_best_val_loss:.4f}")
print(
    f"Validation accuracy at best epoch: "
    f"{do_best_val_accuracy:.4f}"
)

In [ ]:
comparison_exp4_df = pd.DataFrame(
    [
        {
            "Model": "Baseline",
            "Batch size": 32,
            "Class weights": "Yes",
            "Learning rate": 0.001,
            "Dropout": 0.30,
            "Best epoch": best_epoch,
            "Val loss": best_val_loss,
            "Val accuracy": best_val_accuracy,
        },
        {
            "Model": "Batch size 1",
            "Batch size": 1,
            "Class weights": "Yes",
            "Learning rate": 0.001,
            "Dropout": 0.30,
            "Best epoch": bs1_best_epoch,
            "Val loss": bs1_best_val_loss,
            "Val accuracy": bs1_best_val_accuracy,
        },
        {
            "Model": "No class weights",
            "Batch size": 32,
            "Class weights": "No",
            "Learning rate": 0.001,
            "Dropout": 0.30,
            "Best epoch": nw_best_epoch,
            "Val loss": nw_best_val_loss,
            "Val accuracy": nw_best_val_accuracy,
        },
        {
            "Model": "Lower learning rate",
            "Batch size": 32,
            "Class weights": "Yes",
            "Learning rate": 0.0005,
            "Dropout": 0.30,
            "Best epoch": lr_best_epoch,
            "Val loss": lr_best_val_loss,
            "Val accuracy": lr_best_val_accuracy,
        },
        {
            "Model": "Dropout 0.40",
            "Batch size": 32,
            "Class weights": "Yes",
            "Learning rate": 0.001,
            "Dropout": 0.40,
            "Best epoch": do_best_epoch,
            "Val loss": do_best_val_loss,
            "Val accuracy": do_best_val_accuracy,
        },
    ]
)

comparison_exp4_df["Val loss"] = (
    comparison_exp4_df["Val loss"].round(4)
)

comparison_exp4_df["Val accuracy"] = (
    comparison_exp4_df["Val accuracy"].round(4)
)

comparison_exp4_df

### Hyperparameter experiment summary

Four controlled experiments were conducted against the baseline CNN while preserving the fixed training and validation split.

Reducing the training batch size from 32 to 1 resulted in lower validation accuracy and a slightly higher validation loss. Removing class weights produced the same validation accuracy as the baseline but slightly increased validation loss.

Reducing the learning rate from 0.001 to 0.0005 produced the highest validation accuracy of the experiments (84.49%), although its validation loss of 0.4431 was higher than the baseline.

Increasing dropout from 0.30 to 0.40 produced the lowest validation loss, decreasing it from 0.4333 for the baseline to 0.4251, while also slightly increasing validation accuracy from 83.66% to 83.93%.

Since validation loss was used consistently for model checkpointing and model selection, the dropout-0.40 configuration was selected as the strongest candidate for final evaluation. The selected model will next be assessed on the untouched test subset using class-level metrics, including precision, recall, F1-score and the confusion matrix.

## Conclusion

A revised CNN modelling pipeline was developed using the deduplicated and leakage-controlled dataset produced during the revised EDA. The fixed train, validation and test allocation from `dataset_split.csv` was preserved throughout modelling, ensuring that all experiments used the same data split and that the test subset remained untouched.

The baseline CNN used three convolutional blocks, MaxPooling, global average pooling, dropout, a dense classification layer and a four-class softmax output. Moderate geometric augmentation was applied only to the training data, while class weights were used to reduce the effect of the remaining class imbalance.

The baseline model achieved its best validation performance at epoch 15, with a validation loss of 0.4333 and validation accuracy of 83.66%.

Four controlled experiments were then conducted, each changing one modelling choice while keeping the remaining training setup unchanged. Reducing the batch size from 32 to 1 resulted in worse validation performance. Removing class weights produced the same validation accuracy as the baseline but a slightly higher validation loss. Reducing the learning rate from 0.001 to 0.0005 produced the highest validation accuracy of 84.49%, although its validation loss was slightly worse than the baseline.

Increasing the dropout rate from 0.30 to 0.40 produced the lowest validation loss of all tested configurations, decreasing validation loss to 0.4251 while achieving a validation accuracy of 83.93%.

Because validation loss was used consistently for checkpointing and model selection, the dropout-0.40 configuration was selected as the strongest candidate for final evaluation. The selected model will be evaluated on the untouched test subset in the next stage using precision, recall, F1-score and a confusion matrix to assess both overall and class-level performance.